In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:

# STEP 1: Clean and filter the ratings dataset

# Keep only ratings for books available in the books dataset
ratings = ratings[ratings['isbn'].isin(books['isbn'])]

# Remove users who have rated fewer than 200 books
user_rating_counts = ratings['user-id'].value_counts()
ratings = ratings[ratings['user-id'].isin(
    user_rating_counts[user_rating_counts >= 200].index
)]

# Remove books with fewer than 100 ratings
book_rating_counts = ratings['isbn'].value_counts()
ratings = ratings[ratings['isbn'].isin(
    book_rating_counts[book_rating_counts >= 100].index
)]


# STEP 2: Create the user-book rating matrix

book_ratings = ratings.pivot_table(
    index='isbn',
    columns='user-id',
    values='rating'
).fillna(0)


# STEP 3: Train the K-Nearest Neighbors model

from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

model.fit(book_ratings.values)


# STEP 4: Create the recommendation function

def get_recommends(book_title):

    # Find the ISBN of the requested book
    book_isbn = books.loc[
        books['title'] == book_title, 'isbn'
    ].iloc[0]

    # Get the position of the book in the rating matrix
    book_index = book_ratings.index.get_loc(book_isbn)

    # Find the 6 nearest books (including the original book)
    distances, indices = model.kneighbors(
        book_ratings.iloc[book_index].values.reshape(1, -1),
        n_neighbors=6
    )

    recommendations = []

    # Skip the first result because it is the original book
    for i in range(1, len(distances[0])):

        recommended_isbn = book_ratings.index[indices[0][i]]

        recommended_title = books.loc[
            books['isbn'] == recommended_isbn, 'title'
        ].iloc[0]

        recommendations.append([
            recommended_title,
            distances[0][i]
        ])

    # Reverse the recommendations to match the challenge's expected order
    recommendations.reverse()

    return [book_title, recommendations]


In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):


  return recommended_books

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()